# 9. Temporal Alignment Evaluation

Notebook 8 matched individual events to each other. A complementary question: regardless of which events end up
paired, **how precisely do event boundaries line up in time?** `peyes.create_boolean_channel` converts a label or
event sequence into an MNE-style boolean array marking onsets (or offsets); `peyes.channel_metrics`
(`peyes.alignment_metrics`) then measures timing precision directly from two such sequences.

In [1]:
import numpy as np
import peyes
import _helpers

d = _helpers.load_example_trial()
ground_truth = d["raters"]["RA"]
detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
prediction, _ = detector.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)

## Boolean onset/offset channels

`create_boolean_channel(channel_type, data)` accepts either a label sequence or a sequence of `Event` objects;
`channel_type` is `"onset"` (or `"start"`) or `"offset"` (or `"end"`):

In [2]:
gt_onsets = peyes.create_boolean_channel(channel_type="onset", data=ground_truth)
pred_onsets = peyes.create_boolean_channel(channel_type="onset", data=prediction)
print(f"{gt_onsets.sum()} ground-truth onsets, {pred_onsets.sum()} predicted onsets")

43 ground-truth onsets, 62 predicted onsets


## Onset/offset timing differences

`onset_differences` (and `offset_differences`) take the *label/event sequences directly* (they build the boolean
channels internally) and, for each ground-truth onset within `max_diff` samples of a predicted one, return the
signed sample difference:

In [3]:
onset_diffs = peyes.channel_metrics.onset_differences(ground_truth, prediction, max_diff=25)
print(f"{len(onset_diffs)} onsets matched within 25 samples; mean diff = {onset_diffs.mean():.1f} samples")
onset_diffs[:10]

35 onsets matched within 25 samples; mean diff = -0.0 samples


array([ 3,  0, -1, -1,  1, -1,  3, -1, -2,  0])

## Detection metrics across thresholds

`onset_detection_metrics` (and `offset_detection_metrics`) sweep one or more sample-tolerance thresholds and report
a full signal-detection summary (hits, false alarms, precision/recall/F1, d-prime, criterion) at each:

In [4]:
peyes.channel_metrics.onset_detection_metrics(ground_truth, prediction, threshold=[1, 2, 5, 10, 25])

metric,P,PP,TP,N,recall,precision,f1,false_alarm_rate,d_prime,criterion
threshold,,,,,,,,,,
1,43.0,62.0,26.0,885.000000,0.604651,0.419355,0.495238,0.040678,2.008277,0.738734
2,43.0,62.0,30.0,513.800000,0.697674,0.483871,0.571429,0.062281,2.053627,0.509090
5,43.0,62.0,35.0,210.090909,0.813953,0.564516,0.666667,0.128516,2.025995,0.120438
10,43.0,62.0,35.0,89.571429,0.813953,0.564516,0.666667,0.301435,1.412836,-0.186142
25,43.0,62.0,35.0,11.588235,0.813953,0.564516,0.666667,NaN,NaN,NaN


In [5]:
peyes.channel_metrics.offset_detection_metrics(ground_truth, prediction, threshold=[1, 2, 5, 10, 25])

metric,P,PP,TP,N,recall,precision,f1,false_alarm_rate,d_prime,criterion
threshold,,,,,,,,,,
1,43.0,62.0,26.0,885.000000,0.604651,0.419355,0.495238,0.040678,2.008277,0.738734
2,43.0,62.0,31.0,513.800000,0.720930,0.500000,0.590476,0.060335,2.137577,0.483181
5,43.0,62.0,35.0,210.090909,0.813953,0.564516,0.666667,0.128516,2.025995,0.120438
10,43.0,62.0,35.0,89.571429,0.813953,0.564516,0.666667,0.301435,1.412836,-0.186142
25,43.0,62.0,35.0,11.588235,0.813953,0.564516,0.666667,NaN,NaN,NaN


Onset alignment is consistently better than offset alignment here — a common finding, since saccade offsets
(the start of the following fixation) are harder to pin down precisely than onsets.

## What's next

**[10 Visualizing Gaze & Events](./10%20Visualizing%20Gaze%20%26%20Events.ipynb)** — plotting the raw gaze data and
events this guide has been computing all along.